In [4]:
import pandas as pd
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# AWS

In [1]:
import pandas as pd
from pathlib import Path

# ── list the three manifest files ────────────────────────────────
manifest_dir = Path("aws")            # adjust if your path differs
files = [
    manifest_dir / "echo-study-directories.txt",
    manifest_dir / "echo-study-1-directories.txt",
    manifest_dir / "echo-study-2-directories.txt",
]

# ── load & reshape each, then concatenate ───────────────────────
dfs = []
for fp in files:
    raw = pd.read_csv(fp, header=None, sep="\t")     # many-cols / 1-row
    df = (
        raw.T              # cols ➜ rows
          .stack()         # drop NaNs
          .reset_index(drop=True)
          .to_frame(name="s3_key")
    )
    dfs.append(df)

all_studies = pd.concat(dfs, ignore_index=True)

# optional: drop duplicates or inspect
# all_studies = all_studies.drop_duplicates()
# print(all_studies.shape)


In [2]:
# grab the trailing numeric-dot UID (or whatever sits after the last "/")
all_studies["aws_study_id"] = (
    all_studies["s3_key"]
        .str.rstrip("/")          # drop any trailing slash
        .str.split("/")           # split on path separators
        .str[-1]                  # take the last chunk
)

# ── sanity-check ─────────────────────────────
# all_studies.head()


In [5]:
print(len(all_studies))
all_studies.head()

320854


,s3_key,aws_study_id
0,echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285
1,echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703125808.10568238/,1.2.276.0.7230010.3.1.2.1714512485.1.1703125808.10568238
2,echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703138434.10795984/,1.2.276.0.7230010.3.1.2.1714512485.1.1703138434.10795984
3,echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703152147.11023754/,1.2.276.0.7230010.3.1.2.1714512485.1.1703152147.11023754
4,echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703164064.11225415/,1.2.276.0.7230010.3.1.2.1714512485.1.1703164064.11225415


In [60]:
all_studies.to_csv('aws/all_studies.csv')

# Syngo Overlap
How many studies in Syngo are in the DeID key?

In [53]:
import pandas as pd
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [54]:
data_dir = '/cluster/projects/bwanggroup/echo_reports/echo-reports/data/dataset/Lee_Echo_Syngo'
adler = pd.read_csv(os.path.join(data_dir, 'Adler.csv'))
analytics_report = pd.read_csv(os.path.join(data_dir, 'Analytics_Report.csv'))
analytics_study = pd.read_csv(os.path.join(data_dir, 'AnalyticsStudy.csv'))
department = pd.read_csv(os.path.join(data_dir, 'Department.csv'))
field_map = pd.read_csv(os.path.join(data_dir, 'FieldMap.csv'))
measurement_type = pd.read_csv(os.path.join(data_dir, 'MeasurementType.csv'))
modalities = pd.read_csv(os.path.join(data_dir, 'Modalities.csv'))
observations = pd.read_csv(os.path.join(data_dir, 'Observations.csv'))
study_details = pd.read_csv(os.path.join(data_dir, 'StudyDetails.csv'))

/tmp/ipykernel_2427129/3515026001.py:3: DtypeWarning: Columns (18,19,23,24,36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  analytics_report = pd.read_csv(os.path.join(data_dir, 'Analytics_Report.csv'))
/tmp/ipykernel_2427129/3515026001.py:10: DtypeWarning: Columns (6,7,8,16) have mixed types. Specify dtype option on import or set low_memory=False.
  study_details = pd.read_csv(os.path.join(data_dir, 'StudyDetails.csv'))


In [55]:
data_dir = '/cluster/projects/bwanggroup/echo_reports/echo-reports/data/dataset/Lee_Syngo_AnalyticMeasurement'
analytics_measure = pd.read_csv(os.path.join(data_dir, 'AnalyticsMeasure_Total.csv'))

In [66]:
analytics_measure.head()

,StudyRef,MeasurementTypeRef,MeasurementName,Value,Units
0,1000710,126,Height,189.000000,cm
1,1000710,127,Weight,97.300000,kg
2,1000710,140,Regenerate Study Structured Report,1.000000,unitless
3,1000710,10650,Stress Data Persisted,1.000000,unitless
4,1000710,13019,BSA (Boyd),2.265217,m2


In [67]:
len(analytics_measure['StudyRef'].unique())

491644

In [65]:
len(study_details['STUDY_REF'].unique())

389666

# Parts 2 and 3

In [6]:
p2_p3 = pd.read_csv('/cluster/home/t115318uhn/echo-reports/data/aws/R_21_009_011_echo_study_parts2and3_results.csv')

In [7]:
print(p2_p3.shape)
p2_p3.head()

(342763, 7)


,STUDY_REF,PATIENT_ID,STUDY_DATE,STUDY_TIME,ACCESSION_NUMBER,OriginalStudyID,DeidentifiedStudyID
0,1563378,G0357227TSHG,20110811,70641.0,2844100.001TSHG,1.2.124.113532.25.47541.38022.20110810.133108.68626019,1.2.276.0.7230010.3.1.2.845494328.1.1703113032.13338275
1,1070167,4174420,20140318,144646.0,29853237,1.2.124.113532.30763.51996.30715.20140318.141109.3525810,1.2.276.0.7230010.3.1.2.1714578744.1.1703114586.7979678
2,1199431,2368296,20140324,140039.0,29879866,1.2.124.113532.30763.51996.30715.20140324.134336.6750817,1.2.276.0.7230010.3.1.2.1714578744.1.1703115588.7992453
3,1199405,2755844,20140326,93344.0,29889926,1.2.124.113532.30763.51996.30715.20140326.95941.8525045,1.2.276.0.7230010.3.1.2.1714512485.1.1703116173.10393930
4,1067224,4091284,20140417,142030.0,29994381,1.2.124.113532.30763.51996.30715.20140417.121425.5552659,1.2.276.0.7230010.3.1.2.1714578744.1.1703119511.8053048


# Part 1
Which studies actually on AWS are in the DeID key?

In [9]:
syngo_deid = pd.read_csv('/cluster/home/t115318uhn/echo-reports/syngo_deid.csv')

In [10]:
print(len(syngo_deid))
syngo_deid.head()

237022


,DeidentifiedStudyID,OriginalStudyID,STUDY_REF,PATIENT_ID,STUDY_DATE,STUDY_TIME,ACCESSION_NUMBER
0,1.2.276.0.7230010.3.1.2.1714578744.1.1703642763.16086302,1.2.840.113680.1.103.57589.1201541989.392930,1452154,#11-02-003,20080128,93949.0,NaN
1,1.2.276.0.7230010.3.1.2.811753780.1.1703638898.16126008,1.2.840.113680.1.103.57589.1201541989.392930,1452154,#11-02-003,20080128,93949.0,NaN
2,1.2.276.0.7230010.3.1.2.845494328.1.1703200559.14758115,1.2.124.113532.36.59504.16340.20110317.142116.7522070,1157355,2393668,20110317,144215.0,25149762
3,1.2.276.0.7230010.3.1.2.811753780.1.1703538509.15238312,1.2.840.113680.1.103.51602.1120748752.341366,1546905,3233860,20050707,100552.0,NaN
4,1.2.276.0.7230010.3.1.2.845494328.1.1703543619.20977646,1.2.840.113680.1.103.51602.1137682790.772181,1501847,3278818,20060119,95950.0,NaN


# Merged

In [19]:
for df in (p2_p3, syngo_deid):
    df['DeidentifiedStudyID'] = df['DeidentifiedStudyID'].astype(str)

union_deid = (
    pd.concat([p2_p3, syngo_deid], ignore_index=True)
      .drop_duplicates(subset='DeidentifiedStudyID', keep='first')  # keeps the p2_p3 version when duplicated
)

print(union_deid.shape)

(342763, 7)


In [20]:
print(union_deid.shape)
union_deid.head()

(342763, 7)


,STUDY_REF,PATIENT_ID,STUDY_DATE,STUDY_TIME,ACCESSION_NUMBER,OriginalStudyID,DeidentifiedStudyID
0,1563378,G0357227TSHG,20110811,70641.0,2844100.001TSHG,1.2.124.113532.25.47541.38022.20110810.133108.68626019,1.2.276.0.7230010.3.1.2.845494328.1.1703113032.13338275
1,1070167,4174420,20140318,144646.0,29853237,1.2.124.113532.30763.51996.30715.20140318.141109.3525810,1.2.276.0.7230010.3.1.2.1714578744.1.1703114586.7979678
2,1199431,2368296,20140324,140039.0,29879866,1.2.124.113532.30763.51996.30715.20140324.134336.6750817,1.2.276.0.7230010.3.1.2.1714578744.1.1703115588.7992453
3,1199405,2755844,20140326,93344.0,29889926,1.2.124.113532.30763.51996.30715.20140326.95941.8525045,1.2.276.0.7230010.3.1.2.1714512485.1.1703116173.10393930
4,1067224,4091284,20140417,142030.0,29994381,1.2.124.113532.30763.51996.30715.20140417.121425.5552659,1.2.276.0.7230010.3.1.2.1714578744.1.1703119511.8053048


In [21]:
import pandas as pd

# ── 1. make sure both frames have a comparable UID column ───────────
#   • all_studies already has 'aws_study_id' (built earlier)
#   • syngo_deid has 'DeidentifiedStudyID'
#     ── strip any surrounding spaces just in case
union_deid["DeidentifiedStudyID"] = union_deid["DeidentifiedStudyID"].str.strip()

# ── 2. merge on the UID ─────────────────────────────────────────────
aws_uhn = (
    union_deid
        .merge(
            all_studies[["aws_study_id", "s3_key"]],
            left_on="DeidentifiedStudyID",
            right_on="aws_study_id",
            how="inner"          # keep only the overlap
        )
        .drop(columns=["aws_study_id"])   # helper column no longer needed
)

In [27]:
# aws_syngo now has every syngo row that exists in S3,
# plus a new 's3_key' column with the full path.
print(aws_uhn.shape)
aws_uhn.head()

(320854, 9)


,STUDY_REF,PATIENT_ID,STUDY_DATE,STUDY_TIME,ACCESSION_NUMBER,OriginalStudyID,DeidentifiedStudyID,s3_key,study_dir
0,1563378,G0357227TSHG,20110811,70641.0,2844100.001TSHG,1.2.124.113532.25.47541.38022.20110810.133108.68626019,1.2.276.0.7230010.3.1.2.845494328.1.1703113032.13338275,echo-study/1.2.276.0.7230010.3.1.2.845494328.1.1703113032.13338275/,echo-study
1,1070167,4174420,20140318,144646.0,29853237,1.2.124.113532.30763.51996.30715.20140318.141109.3525810,1.2.276.0.7230010.3.1.2.1714578744.1.1703114586.7979678,echo-study/1.2.276.0.7230010.3.1.2.1714578744.1.1703114586.7979678/,echo-study
2,1199431,2368296,20140324,140039.0,29879866,1.2.124.113532.30763.51996.30715.20140324.134336.6750817,1.2.276.0.7230010.3.1.2.1714578744.1.1703115588.7992453,echo-study/1.2.276.0.7230010.3.1.2.1714578744.1.1703115588.7992453/,echo-study
3,1199405,2755844,20140326,93344.0,29889926,1.2.124.113532.30763.51996.30715.20140326.95941.8525045,1.2.276.0.7230010.3.1.2.1714512485.1.1703116173.10393930,echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116173.10393930/,echo-study
4,1067224,4091284,20140417,142030.0,29994381,1.2.124.113532.30763.51996.30715.20140417.121425.5552659,1.2.276.0.7230010.3.1.2.1714578744.1.1703119511.8053048,echo-study/1.2.276.0.7230010.3.1.2.1714578744.1.1703119511.8053048/,echo-study


In [26]:
# add a column holding the top-level study directory
aws_uhn['study_dir'] = aws_uhn['s3_key'].str.partition('/')[0]   # echo-study, echo-study-1, echo-study-2

# count rows per directory
counts = aws_uhn['study_dir'].value_counts().sort_index()

print(counts)

study_dir
echo-study      215113
echo-study-1     26144
echo-study-2     79597
Name: count, dtype: int64


In [25]:
aws_uhn.to_csv('aws/aws_uhn.csv')

# HeartLab

In [37]:
data_dir = '/cluster/projects/bwanggroup/echo_reports/echo-reports/data/dataset'

In [41]:
patients = pd.read_csv(os.path.join(data_dir,'Patients_No_PHI.csv'))
reports = pd.read_csv(os.path.join(data_dir,'REPORTS.csv'))
series = pd.read_csv(os.path.join(data_dir,'SERIES.csv'))
studies = pd.read_csv(os.path.join(data_dir,'STUDIES.csv'))

/tmp/ipykernel_2427129/1450941421.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  reports = pd.read_csv(os.path.join(data_dir,'REPORTS.csv'))
/tmp/ipykernel_2427129/1450941421.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  series = pd.read_csv(os.path.join(data_dir,'SERIES.csv'))
/tmp/ipykernel_2427129/1450941421.py:4: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  studies = pd.read_csv(os.path.join(data_dir,'STUDIES.csv'))


In [46]:
print(len(studies))
studies.head()

431710


,ID,PATI_ID,ACCESSION_NUMBER,REFERRING_PHYSICIANS_NAME,STUDY_DATE,STUDY_DESCRIPTION,STUDY_ID,STUDY_INSTANCE_UID,STUDY_TIME,STUDY_STATE,STUDY_SOURCE,ADM_ID,DICTATIONS_DATE,DICATATION_DATE,DICTATION_DATE,FLAG,RECONCILED_DATE
0,421,1071,NaN,NaN,10/17/02 08:09:11,NaN,NaN,1.2.840.113543.6.6.1.2.4.150521856054391.20228929351,80911.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,422,1072,NaN,NaN,10/17/02 08:00:38,UHN1,NaN,1.2.840.113680.1.103.60427.1034870438.961521,80038.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,423,1073,NaN,NaN,10/17/02 08:32:10,NaN,NaN,1.2.840.113543.6.6.1.1.4.2415947926921624359.20228930730,83210.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,424,1074,NaN,NaN,10/17/02 08:26:25,NaN,NaN,1.2.840.113543.6.6.1.1.4.24159236081775595303.20228930385,82625.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,425,1075,NaN,NaN,10/17/02 08:19:25,Routine,NaN,1.2.840.113680.4.103.71215.20021017081925,81925.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [81]:
# ── ensure both columns we compare are plain strings ──────────────
union_deid["DeidentifiedStudyID"] = (
    union_deid["DeidentifiedStudyID"].astype(str).str.strip()
)
all_studies["aws_study_id"] = (
    all_studies["aws_study_id"].astype(str).str.strip()
)

# ── keep only Syngo rows whose UID exists in all_studies ──────────
syngo_deid_aws = syngo_deid[
    syngo_deid["DeidentifiedStudyID"].isin(all_studies["aws_study_id"])
].copy()

print(syngo_deid_aws.shape)   # rows kept / columns unchanged
syngo_deid_aws.head()


(215113, 7)


,DeidentifiedStudyID,OriginalStudyID,STUDY_REF,PATIENT_ID,STUDY_DATE,STUDY_TIME,ACCESSION_NUMBER
0,1.2.276.0.7230010.3.1.2.1714578744.1.1703642763.16086302,1.2.840.113680.1.103.57589.1201541989.392930,1452154,#11-02-003,20080128,93949.0,NaN
2,1.2.276.0.7230010.3.1.2.845494328.1.1703200559.14758115,1.2.124.113532.36.59504.16340.20110317.142116.7522070,1157355,2393668,20110317,144215.0,25149762
3,1.2.276.0.7230010.3.1.2.811753780.1.1703538509.15238312,1.2.840.113680.1.103.51602.1120748752.341366,1546905,3233860,20050707,100552.0,NaN
4,1.2.276.0.7230010.3.1.2.845494328.1.1703543619.20977646,1.2.840.113680.1.103.51602.1137682790.772181,1501847,3278818,20060119,95950.0,NaN
5,1.2.276.0.7230010.3.1.2.1714578744.1.1703561546.15322813,1.2.840.113680.1.103.51602.1209402194.673271,1479224,3697059,20080428,90314.0,NaN


In [82]:
# ── tidy the key columns so they compare cleanly ──────────────────
studies["STUDY_INSTANCE_UID"] = (
    studies["STUDY_INSTANCE_UID"].astype(str).str.strip()
)
syngo_deid_aws["OriginalStudyID"] = (
    syngo_deid_aws["OriginalStudyID"].astype(str).str.strip()
)

# ── keep only studies whose UID is present in Syngo’s list ────────
studies_syngo = studies[
    studies["STUDY_INSTANCE_UID"].isin(syngo_deid_aws["OriginalStudyID"])
].copy()

print(studies_syngo.shape)   # rows that overlap / same columns as `studies`
studies_syngo.head()


(203683, 17)


,ID,PATI_ID,ACCESSION_NUMBER,REFERRING_PHYSICIANS_NAME,STUDY_DATE,STUDY_DESCRIPTION,STUDY_ID,STUDY_INSTANCE_UID,STUDY_TIME,STUDY_STATE,STUDY_SOURCE,ADM_ID,DICTATIONS_DATE,DICATATION_DATE,DICTATION_DATE,FLAG,RECONCILED_DATE
123,287,916,NaN,NaN,10/11/02 09:11:38,NaN,NaN,1.2.840.113663.1298.56555945.1.3383.20021011.1091138,91138.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137,301,930,NaN,NaN,10/11/02 15:02:14,NaN,NaN,1.2.840.113663.1298.56555945.1.9300.20021011.1150214,150214.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22529,27370,32451,NaN,NaN,02/10/04 12:31:30,zzz,NaN,1.2.840.113619.2.98.1722.1076392656.0.953,123130.0,NaN,"IMAGE,USER",NaN,NaN,NaN,NaN,NaN,NaN
24064,31754,40624,NaN,NaN,04/08/04 08:43:45,NaN,NaN,1.2.840.113543.6.6.2.0.97083304982.20040408084150,84345.0,NaN,"IMAGE,USER",NaN,NaN,NaN,NaN,NaN,NaN
24254,27932,19686,NaN,NaN,02/18/04 11:03:46,NaN,NaN,1.2.840.113543.6.6.2.0.37283301485.20040218110000,110346.0,NaN,"IMAGE,USER",NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
print(aws_syngo.shape)
aws_syngo.head()

(215113, 10)


,DeidentifiedStudyID,OriginalStudyID,STUDY_REF,PATIENT_ID,STUDY_DATE,STUDY_TIME,ACCESSION_NUMBER,s3_key,study_uid,study_dir
0,1.2.276.0.7230010.3.1.2.1714578744.1.1703642763.16086302,1.2.840.113680.1.103.57589.1201541989.392930,1452154,#11-02-003,20080128,93949.0,NaN,echo-study/1.2.276.0.7230010.3.1.2.1714578744.1.1703642763.16086302/,1.2.276.0.7230010.3.1.2.1714578744.1.1703642763.16086302,echo-study
1,1.2.276.0.7230010.3.1.2.845494328.1.1703200559.14758115,1.2.124.113532.36.59504.16340.20110317.142116.7522070,1157355,2393668,20110317,144215.0,25149762,echo-study/1.2.276.0.7230010.3.1.2.845494328.1.1703200559.14758115/,1.2.276.0.7230010.3.1.2.845494328.1.1703200559.14758115,echo-study
2,1.2.276.0.7230010.3.1.2.811753780.1.1703538509.15238312,1.2.840.113680.1.103.51602.1120748752.341366,1546905,3233860,20050707,100552.0,NaN,echo-study/1.2.276.0.7230010.3.1.2.811753780.1.1703538509.15238312/,1.2.276.0.7230010.3.1.2.811753780.1.1703538509.15238312,echo-study
3,1.2.276.0.7230010.3.1.2.845494328.1.1703543619.20977646,1.2.840.113680.1.103.51602.1137682790.772181,1501847,3278818,20060119,95950.0,NaN,echo-study/1.2.276.0.7230010.3.1.2.845494328.1.1703543619.20977646/,1.2.276.0.7230010.3.1.2.845494328.1.1703543619.20977646,echo-study
4,1.2.276.0.7230010.3.1.2.1714578744.1.1703561546.15322813,1.2.840.113680.1.103.51602.1209402194.673271,1479224,3697059,20080428,90314.0,NaN,echo-study/1.2.276.0.7230010.3.1.2.1714578744.1.1703561546.15322813/,1.2.276.0.7230010.3.1.2.1714578744.1.1703561546.15322813,echo-study


In [85]:
# ── count the overlap ──────────────────────────────────────────────
overlap_count = (
    studies_syngo["STUDY_INSTANCE_UID"]
        .isin(aws_syngo["OriginalStudyID"])
        .sum()                      # True → 1, False → 0
)

print("rows in studies_syngo that match aws_syngo:", overlap_count)

# optional: confirm that’s every row (or inspect the misses)
# print(overlap_count, "/", len(studies_syngo))
# studies_syngo[~studies_syngo["STUDY_INSTANCE_UID"].isin(aws_syngo["OriginalStudyID"])].head()


rows in studies_syngo that match aws_syngo: 203683


### In other words, aws_syngo is the superset.
### Still, some measurements / findings are in HeartLab.